In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - INGEST CONSTRUCTORS
# ============================================================
# Purpose:
# Reads constructors.json from S3 Landing, adds ingestion
# metadata, and writes the data in Delta format to Bronze.
# ============================================================


# COMMAND ----------

# Load environment configuration

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

# Load Bronze helper functions

# MAGIC %run "../00_common/02_bronze_helpers"


# COMMAND ----------

# Source and target paths

source_file = f"{s3_root_path}/landing/constructors.json"

target_path = f"{s3_root_path}/Bronze/Kalashri/constructors"

print("Source File :", source_file)
print("Target Path :", target_path)


# COMMAND ----------

# Define schema

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

constructors_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])


# COMMAND ----------

# Read constructors JSON from S3 Landing

constructors_df = (
    spark.read.format('json')
    .format("json")
    .schema(constructors_schema)
    .load(source_file)
)

display(constructors_df)


# COMMAND ----------

# Check schema

constructors_df.printSchema()


# COMMAND ----------

# Add ingestion metadata

constructors_final_df = add_ingestion_metadata(constructors_df)

display(constructors_final_df)


# COMMAND ----------

# Write to S3 Bronze in Delta format

(
    constructors_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(target_path)
)


# COMMAND ----------

# Validate Bronze Delta data

bronze_constructors_df = (
    spark.read
    .format("delta")
    .load(target_path)
)

display(bronze_constructors_df)


# COMMAND ----------

# Validate record count

print("Total Bronze Records:", bronze_constructors_df.count())

print("Bronze Constructors data successfully written to:")
print(target_path)

In [0]:
%sql

SELECT *
FROM delta.`s3://bizf1project123/Bronze/Kalashri/constructors`;